# Fase 2: Feature Engineering y Modelo Baseline
## Extracción de Características Temporales y Validación con StratifiedGroupKFold

**Autor:** Íñigo Guzmán  

### Principio de Rigor Metodológico: Prevención de Data Leakage
En datos de mecanizado CNC, las series temporales de un mismo experimento están autocorrelacionadas. Si se realizara una partición aleatoria por filas, el modelo memorizaría el contexto específico del bloque de cera o de la vibración del banco de pruebas, dando un accuracy artificialmente alto (>95%) que fracasaría en producción.

Para garantizar resultados reales y honestos, se utiliza **`StratifiedGroupKFold`** agrupando por el ID de experimento (`exp_id`), evaluando el modelo siempre en piezas y experimentos completamente inéditos.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import numpy as np
from src.features import construir_dataset_segmentado
from src.baseline import ejecutar_evaluacion_baseline

data_dir = Path("../data/raw")
df_feats = construir_dataset_segmentado(data_dir)
print(f"Dimensiones del dataset procesado: {df_feats.shape}")
df_feats.head()

### 2. Resultados de los Modelos Baseline
Evaluamos tres baselines de referencia:
1. **Dummy (Clase Mayoritaria):** Clasificador trivial que siempre predice 'worn'.
2. **Logistic Regression:** Modelo lineal estándar con estandarización de características.
3. **Random Forest Baseline:** Bosque aleatorio de 100 árboles sin ajuste fino de hiperparámetros.

In [ ]:
reports_dir = Path("../reports")
res = ejecutar_evaluacion_baseline(data_dir, reports_dir)

df_res = pd.DataFrame(res).T
df_res[["accuracy_mean", "accuracy_std", "f1_mean", "roc_auc_mean"]]

### 3. Conclusión de la Fase Baseline
La evaluación con `StratifiedGroupKFold` muestra que predecir el desgaste en experimentos no vistos es un reto complejo debido a las variaciones de `feedrate` (3, 6, 12, 15, 20 mm/s) y `clamp_pressure` (2.5, 3, 4 bar).
En la Fase 3, implementaremos un modelo con **XGBoost**, selección de variables de corte más discriminativas e interpretabilidad con **SHAP**.